# SPLITTING

In [ ]:
from zipper_owntry import Zipp3r

In [ ]:
image_in = '/app/data/data1channel-001.tif'
output_path = '/app/Zipping-pruebas/OUTPUT'

In [ ]:
zipper = Zipp3r(image_in, output_path)

# Create a dictionary containing information on all blocks
zipper.set_block_slices(
    image_shape=[146, 3788, 3788],
    blocksize=[146, 947, 947],
    margins=[0,30,30]
)

In [ ]:
# Cut the image according to the block_slices
zipper.split_image()

# STITCHING

In [ ]:
import os
import re
import numpy as np
import tifffile

In [ ]:
def load_tiff(file_path):
    """Carga un archivo TIFF como un array numpy."""
    with tifffile.TiffFile(file_path) as tif:
        return tif.asarray()

In [ ]:
def parse_filename(filename):
    """Extrae las coordenadas Z, X, Y desde el nombre del archivo."""
    pattern = r'_block(\d+)x(\d+)x(\d+)\.tif$'
    match = re.search(pattern, filename)
    if match:
        return int(match.group(1)), int(match.group(2)), int(match.group(3))
    else:
        raise ValueError(f"Filename {filename} does not match the expected pattern.")

In [ ]:
def stitch_tiles(tiles, overlap_x, overlap_y):
    """Realiza el stitching de los tiles considerando el overlapping."""

    n_tiles_y = len(tiles)
    n_tiles_x = len(tiles[0])

    # Determinar el tamaño máximo de los tiles
    max_tile_y = max(max(tile.shape[1] for tile in row) for row in tiles)
    max_tile_x = max(max(tile.shape[2] for tile in row) for row in tiles)

    # Calcular el tamaño de la imagen final
    stitched_y = max_tile_y * n_tiles_y - overlap_y * (n_tiles_y - 1)
    stitched_x = max_tile_x * n_tiles_x - overlap_x * (n_tiles_x - 1)
    stitched_image = np.zeros((tiles[0][0].shape[0], stitched_y, stitched_x), dtype=np.float32)

    for i in range(n_tiles_y):
        for j in range(n_tiles_x):
            tile = tiles[i][j]
            tile_z, tile_y, tile_x = tile.shape

            # Calcular overlap para este tile
            left_overlap = overlap_x if j > 0 else 0
            right_overlap = overlap_x if j < n_tiles_x - 1 else 0
            top_overlap = overlap_y if i > 0 else 0
            bottom_overlap = overlap_y if i < n_tiles_y - 1 else 0

            start_y = i * (max_tile_y - top_overlap)
            start_x = j * (max_tile_x - left_overlap)
            end_y = start_y + tile_y
            end_x = start_x + tile_x

            # Asegurarse de que los índices estén dentro del rango de la imagen final
            end_y = min(end_y, stitched_y)
            end_x = min(end_x, stitched_x)
            start_y = max(start_y, 0)
            start_x = max(start_x, 0)

            # Asegurarse de que no se salga del rango del tile
            tile_end_y = min(tile_y, end_y - start_y)
            tile_end_x = min(tile_x, end_x - start_x)

            # Cortar el tile para evitar superposición
            tile_cut = tile[:, :tile_end_y, :tile_end_x]

            # Sumar la intensidad del tile a la imagen final
            stitched_image[:, start_y:end_y, start_x:end_x] += tile_cut

    return stitched_image.astype(np.uint16)

In [ ]:
def process_directory(directory, overlap_x, overlap_y):
    """Lee todos los archivos TIFF en el directorio y realiza el stitching."""
    tile_dict = {}

    # Recorre todos los archivos en el directorio
    for filename in os.listdir(directory):
        if filename.endswith(".tif"):
            z, x, y = parse_filename(filename)
            file_path = os.path.join(directory, filename)
            tile = load_tiff(file_path)

            if z not in tile_dict:
                tile_dict[z] = []
            tile_dict[z].append((x, y, tile))

    # Ordenar los tiles por sus coordenadas X, Y
    for z in tile_dict:
        tile_dict[z] = sorted(tile_dict[z], key=lambda item: (item[0], item[1]))

    # Determinar dimensiones de la matriz de tiles
    z_levels = sorted(tile_dict.keys())
    stitched_tiles = []
    for z in z_levels:
        # Determinar el tamaño de la matriz de tiles
        max_x = max(x for x, _, _ in tile_dict[z]) + 1
        max_y = max(y for _, y, _ in tile_dict[z]) + 1
        tile_matrix = [[None] * max_y for _ in range(max_x)]

        for x, y, tile in tile_dict[z]:
            tile_matrix[x][y] = tile

        stitched_tiles.append(stitch_tiles(tile_matrix, overlap_x, overlap_y))

    return stitched_tiles

In [ ]:
# Ejemplo de uso
directory = "/app/Zipping-pruebas/OUTPUT/blocks"  # Cambia esto por la ruta de tu directorio
overlap_x = int(30 * 4 / 3) + 10  # Cambia esto según tu configuración
overlap_y = int(30 * 4 / 3) + 10  # Cambia esto según tu configuración

In [ ]:
# Procesa el directorio y realiza el stitching
stitched_images = process_directory(directory, overlap_x, overlap_y)

In [ ]:
# Guardar las imágenes unidas por cada nivel Z como TIFF
for i, stitched_image in enumerate(stitched_images):
    output_path = f'/app/stitched_image_z{overlap_x}.tiff'
    tifffile.imwrite(output_path, stitched_image)